# Análise Prescritiva — Plano executável de recuperação de margem

Este notebook transforma achados diagnósticos em regras e ações prescritivas (guardrails, política de desconto, regras de frete e prioridades de reprecificação).

Objetivo: reduzir pedidos com prejuízo e elevar a margem consolidada atuando exatamente nos recortes que mais drenam lucro.

In [ ]:
import pandas as pd
import numpy as np
import unicodedata
from pathlib import Path

def norm_ascii(s: str) -> str:
    s = '' if s is None else str(s)
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('ascii')
    return ' '.join(s.split()).lower()

DATA_PATH = Path('..') / 'data' / 'raw' / 'Case 1 - Case Retail Store Sales Data.xlsx'
df = pd.read_excel(DATA_PATH, sheet_name='Sales_retail_store')

rename = {}
for c in df.columns:
    k = norm_ascii(c)
    if k == 'regiao':
        rename[c] = 'Região'
    if k == 'preco unitario':
        rename[c] = 'Preço Unitário'
if rename:
    df = df.rename(columns=rename)

df['Data da Venda'] = pd.to_datetime(df['Data da Venda'])
df['Ano'] = df['Data da Venda'].dt.year
for c in ['Faturamento', 'Lucro', 'Desconto', 'Custo de Envio']:
    df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0)

df.shape

## 1) Guardrails (regras obrigatórias)
Regras de aprovação para impedir vendas com prejuízo e padronizar margem mínima por pedido.

In [ ]:
order = df.groupby('Order ID', as_index=False).agg(fat=('Faturamento','sum'), lucro=('Lucro','sum'))
neg = order[order['lucro'] < 0]
share_neg_orders = len(neg)/max(1,len(order))
share_neg_fat = neg['fat'].sum() / (order['fat'].sum() or 1)

pd.DataFrame({
    'Pedidos totais': [len(order)],
    '% pedidos com prejuízo': [share_neg_orders],
    '% faturamento em pedidos com prejuízo': [share_neg_fat],
})

### Proposta de guardrails
- Bloquear automaticamente pedidos com **lucro < 0** (ou exigir aprovação).
- Exigir margem mínima por pedido:
  - **≥10%** para pedidos com envio **Aéreo**.
  - **≥12%** para pedidos com envio **Transporte Rodoviário**.

A seguir, priorizamos onde aplicar regras mais duras por impacto e relevância.

## 2) Prioridades de intervenção (2026)
Aqui listamos subcategorias e combinações (subcategoria × envio) com maior drenagem de lucro em 2026.

In [ ]:
base26 = df[df['Ano']==2026].copy()
fat26 = base26['Faturamento'].sum() or 1

sub = (
    base26.groupby(['Categoria do Produto','Sub-Categoria do Produto'], as_index=False)
          .agg(faturamento=('Faturamento','sum'), lucro=('Lucro','sum'))
)
sub['margem'] = np.where(sub['faturamento']!=0, sub['lucro']/sub['faturamento'], np.nan)
sub['share_2026'] = sub['faturamento']/fat26
sub.sort_values('lucro').head(15)

In [ ]:
combo = (
    base26.groupby(['Categoria do Produto','Sub-Categoria do Produto','Forma de Envio'], as_index=False)
          .agg(faturamento=('Faturamento','sum'), lucro=('Lucro','sum'))
)
combo['margem'] = np.where(combo['faturamento']!=0, combo['lucro']/combo['faturamento'], np.nan)
combo['share_2026'] = combo['faturamento']/fat26

combo[combo['share_2026'] >= 0.01].sort_values('lucro').head(20)

## 3) Matriz prescritiva (regras por foco)
Preencha/ajuste esta matriz com as regras a serem implementadas no ERP/CRM/Checkout:

- **Desconto máximo** por subcategoria.
- **Regras de envio** (ex.: Rodoviário somente com frete repassado e margem mínima).
- **Reprecificação** (quando necessário) para retirar itens do prejuízo.

In [ ]:
matriz = pd.DataFrame([
    {
        'Onde (Categoria > Subcategoria)': 'Mobiliário > Mesas',
        'Regra de desconto': 'Desconto máximo 0%',
        'Regra de envio': 'Rodoviário somente com frete repassado e margem mínima ≥12%',
        'Ação de preço': 'Reprecificar para remover margem negativa',
    },
    {
        'Onde (Categoria > Subcategoria)': 'Mobiliário > Estantes',
        'Regra de desconto': 'Desconto máximo 0%',
        'Regra de envio': 'Rodoviário somente com frete repassado e margem mínima ≥12%',
        'Ação de preço': 'Reprecificar para remover margem negativa',
    },
    {
        'Onde (Categoria > Subcategoria)': 'Material de Escritório > Armazenamento e Organização',
        'Regra de desconto': 'Eliminar faixa 5–10% (máximo 0–5% com guardrail)',
        'Regra de envio': 'Aéreo Rápido somente com frete repassado e margem mínima ≥12%',
        'Ação de preço': 'Ajustar preço efetivo (preço/desconto/frete)',
    },
    {
        'Onde (Categoria > Subcategoria)': 'Tecnologia > Máquinas de Escritório',
        'Regra de desconto': 'Eliminar 5–10% (máximo 0–5% com guardrail)',
        'Regra de envio': 'Evitar subsídio em Rodoviário; exigir margem mínima',
        'Ação de preço': 'Reprecificar seletivamente ou reduzir desconto',
    },
    {
        'Onde (Categoria > Subcategoria)': 'Tecnologia > Periféricos',
        'Regra de desconto': 'Capar em 0–5% com guardrail',
        'Regra de envio': 'Evitar subsídio em Rodoviário; exigir margem mínima',
        'Ação de preço': 'Manter competitividade sem degradar margem',
    },
])
matriz